In [2]:
!pip install mlflow

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 4.2 MB/s eta 0:00:03
   ---------- ----------------------------- 3.1/12.6 MB 5.6 MB/s eta 0:00:02
   --------------- ------------------------ 5.0/12.6 MB 6.6 MB/s eta 0:00:02
   ---------------------- ----------------- 7.1/12.6 MB 7.2 MB/s eta 0:00:01
   ---------------------------- ----------- 8.9/12.6 MB 7.5 MB/s eta 0:00:01
   ---------------------------------- ----- 10.7/12.6 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.6 MB 7.8 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 7.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   --------------------- ------------------ 1.8/3.5 MB 9.1 MB/s eta 0:00:01
   ------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.6 which is incompatible.


In [4]:
import mlflow
import mlflow.sklearn
import numpy as np
import pickle
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [5]:
mlflow.set_experiment("Tuning_Model_PKL_Kosin")

2026/06/27 22:09:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/27 22:09:50 INFO mlflow.store.db.utils: Updating database tables
2026/06/27 22:09:53 INFO mlflow.tracking.fluent: Experiment with name 'Tuning_Model_PKL_Kosin' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:d:/tugas/codes/Machine Learning/AoL Machine '
 'Learning/prediksi_harga_kos/tugas_mlflow/mlruns/1'), creation_time=1782572993214, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782572993214, lifecycle_stage='active', name='Tuning_Model_PKL_Kosin', tags={}, trace_location=None, workspace='default'>

In [7]:
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

In [ ]:
with open('..\\models\\model_kos.pkl', 'rb') as f:
    model = pickle.load(f)

In [12]:
hyperparameter_options = [
    {"n_estimators": 50, "max_depth": 5, "random_state": 42},
    {"n_estimators": 100, "max_depth": 10, "random_state": 42}, 
    {"n_estimators": 150, "max_depth": 15, "random_state": 42}
]

In [13]:
exp_id = None

for i, params in enumerate(hyperparameter_options, start=1):
    run_name = f"Tuning_Run_{i}_depth_{params['max_depth']}"
    
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params)
        
        tuned_model = model.set_params(**params)
        
        tuned_model.fit(X_train, y_train)
        
        model_info = mlflow.sklearn.log_model(
            sk_model=tuned_model, 
            artifact_path="model"
        )
        
        y_pred_log = tuned_model.predict(X_test)
        
        r2 = r2_score(y_test, y_pred_log)
        mlflow.log_metric("R2_Score_Log", r2)
        
        y_test_original = np.expm1(y_test)
        y_pred_original = np.expm1(y_pred_log)
        
        mae = mean_absolute_error(y_test_original, y_pred_original)
        mse = mean_squared_error(y_test_original, y_pred_original)
        rmse = np.sqrt(mse) 
        
        mlflow.log_metric("MAE_Rupiah", mae)
        mlflow.log_metric("MSE_Rupiah", mse)
        mlflow.log_metric("RMSE_Rupiah", rmse)
        
        mlflow.set_tag("Model Base", "Pre-trained PKL")
        mlflow.set_tag("Tuning Status", f"Eksperimen Hyperparameter ke-{i}")
        
        print(f"Run {i} Tuning dengan n_estimators: {params['n_estimators']} , max_depth: {params['max_depth']}")
        print(f"R2 Score (Log) : {r2:.4f}")
        print(f"MAE (Rupiah)   : Rp {mae:,.0f}".replace(",", "."))
        print(f"MSE (Rupiah)   : Rp {mse:,.0f}".replace(",", "."))
        print(f"RMSE (Rupiah)  : Rp {rmse:,.0f}\n".replace(",", "."))
        
        exp_id = run.info.experiment_id

2026/06/27 22:25:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 1 Tuning dengan n_estimators: 50 , max_depth: 5
R2 Score (Log) : 0.7644
MAE (Rupiah)   : Rp 280.056
MSE (Rupiah)   : Rp 179.807.643.447
RMSE (Rupiah)  : Rp 424.037



2026/06/27 22:26:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 2 Tuning dengan n_estimators: 100 , max_depth: 10
R2 Score (Log) : 0.8086
MAE (Rupiah)   : Rp 246.102
MSE (Rupiah)   : Rp 150.502.644.480
RMSE (Rupiah)  : Rp 387.947



2026/06/27 22:26:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 3 Tuning dengan n_estimators: 150 , max_depth: 15
R2 Score (Log) : 0.8086
MAE (Rupiah)   : Rp 242.404
MSE (Rupiah)   : Rp 149.924.307.856
RMSE (Rupiah)  : Rp 387.201



In [14]:
df_runs = mlflow.search_runs(experiment_ids=[exp_id])

print("SUMMARY TUNING RUNS DI MLFLOW DASHBOARD")
summary_cols = [
    'run_id', 
    'params.n_estimators', 
    'params.max_depth', 
    'metrics.R2_Score_Log', 
    'metrics.MAE_Rupiah',
    'metrics.MSE_Rupiah',
    'metrics.RMSE_Rupiah'
]
print(df_runs[summary_cols].to_string(index=False))

print("Me-load model hasil tuning terakhir dari MLflow")
loaded_model = mlflow.sklearn.load_model(model_info.model_uri)
print("Model berhasil di-load")

SUMMARY TUNING RUNS DI MLFLOW DASHBOARD
                          run_id params.n_estimators params.max_depth  metrics.R2_Score_Log  metrics.MAE_Rupiah  metrics.MSE_Rupiah  metrics.RMSE_Rupiah
3a3e56866b734b3d94987cdb60144622                 150               15              0.808627       242403.573700        1.499243e+11        387200.604153
4a39e75635cb4b9e9f38ee31b74b2528                 100               10              0.808554       246101.730475        1.505026e+11        387946.703143
b911a15999234e3e975c054d56a3f9f3                  50                5              0.764435       280056.213853        1.798076e+11        424037.313744
Me-load model hasil tuning terakhir dari MLflow
Model berhasil di-load
